# BigQuery Conversational Analytics (BQCA) Prompt & Response Logging
### Customer Analytics, FinOps & Evaluation Starter Notebook (`bigquery-agent-analytics` v0.5.4)

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/blob/main/examples/bqca_prompt_response_logging_customer_notebook.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/colab-logo.png" alt="Colab logo"> Run in Colab
    </a>
  </td>
  <td>
    <a href="https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/blob/main/examples/bqca_prompt_response_logging_customer_notebook.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/github-logo.png" width="32" alt="GitHub logo"> View Notebook on GitHub
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/main/examples/bqca_prompt_response_logging_customer_notebook.ipynb">
      <img src="https://www.gstatic.com/images/branding/product/1x/google_cloud_48dp.png" alt="Vertex AI logo" width="32"> Open in Vertex AI Workbench
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/blob/main/examples/bqca_prompt_response_logging_customer_notebook.ipynb">
      <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTW1gvOovVlbZAIZylUtf5Iu8-693qS1w5NJw&s" alt="BQ logo" width="35"> Open in BQ Studio
    </a>
  </td>
</table>

<br clear="all"/>

When you enable **Prompt and Response Logging** in **BigQuery Conversational Analytics (BQCA)**, BigQuery streams conversation events into an `agent_events` table in your dataset using the same 17-column schema consumed by the open-source [BigQuery Agent Analytics SDK](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK) (`bigquery-agent-analytics`).

This starter notebook is a **copy-and-run production playbook** for BQCA customers. Point the configuration parameters in **Section 1.1** at your own `PROJECT_ID`, `DATASET_ID`, `RAW_TABLE_ID` (`agent_events`), and `VERTEX_CONNECTION_ID`, and run all cells from top to bottom. For IAM setup, Vertex AI Cloud Resource Connection commands, and a 10-issue troubleshooting runbook, see the companion **[Customer Self-Hosting & Self-Debugging Manual](../docs/guides/bqca-prompt-response-logging-manual.md)**.

**What this notebook sets up and runs for you**

| Section | Customer workflow | What it provisions or runs | BQCA events used |
|---|---|---|---|
| **1** | **Setup, Compatibility Views & `doctor()`** | Installs the SDK if needed, creates two zero-copy BigQuery views (`v_bqca_customer_sdk_events` for the SDK and `v_bqca_customer_turns` for SQL/BI dashboards), runs `client.doctor()`, and pings `AI.GENERATE` (`gemini-3.5-flash`). | All 9 BQCA event types |
| **2** | **Prompt & Response Conversation Explorer** | Summarises conversation volume, answer rates, clarifying questions, fast-path hits and latency by agent and user cohort, and previews paired prompt/response turns. | `USER_MESSAGE_RECEIVED`, `AGENT_RESPONSE`, `INVOCATION_COMPLETED`, `LLM_RESPONSE` |
| **3** | **Customer Sentiment & Frustrated-Turn Triage** | Scores every session with `LLMAsJudge.sentiment()` and classifies user tone with `evaluate_categorical()` (`Positive_Satisfied`, `Neutral_Analytical`, `Clarification_Needed`, `Frustrated_Impatient`), joining scores back to the exact prompts and answers. | `USER_MESSAGE_RECEIVED`, `AGENT_RESPONSE` |
| **4** | **User Question Categorisation & Topic Taxonomy** | Classifies user questions into your business topic taxonomy using both fast bulk mode (`AI.CLASSIFY`) and audited rationale mode (`AI.GENERATE`). | `USER_MESSAGE_RECEIVED` |
| **5** | **Semantic Drift & Golden Question Bank Coverage** | Bootstraps or loads `bqca_golden_qa`, runs `client.drift_detection()`, and runs a production-centric `AI.EMBED` + `ML.DISTANCE` query to find which user questions are outside your Golden Question Bank. | `USER_MESSAGE_RECEIVED` + Golden Q&A table |
| **6** | **Response-Quality Evaluation (Golden Q&A Key & SDK Judges)** | Grades answers against Golden Q&A pairs via `AI.EMBED` + `AI.GENERATE`, and runs `LLMAsJudge.correctness()`, `LLMAsJudge.hallucination()`, `GraderPipeline`, and multi-trial judge consistency (`TrialRunner` + `BQCAPerformanceEvaluator`). | `USER_MESSAGE_RECEIVED`, `AGENT_RESPONSE` + Golden Q&A table |
| **7** | **FinOps, Latency SLOs, Thinking Tokens & Fast-Path ROI** | Evaluates session SLO budgets (`SystemEvaluator`), accounts for Gemini thinking tokens (`thoughts_token_count`), and compares Verified Queries fast-path turns (`attributes.fast_path = true`) against standard NL2SQL turns. | `LLM_RESPONSE`, `INVOCATION_COMPLETED`, `AGENT_RESPONSE` |
| **8** | **Operational Health Monitor & Executive Summary** | Monitors model cut-offs (`finish_reason != 'STOP'`), sanitised `*_ERROR` rows (`bqaa.ERROR_SQL_PREDICATE`), and polite out-of-scope refusals, then prints a live **Customer Production Health Summary Table**. | `LLM_RESPONSE`, `INVOCATION_ERROR`, `AGENT_ERROR`, `LLM_ERROR`, `AGENT_RESPONSE` |

> **What BQCA logs by design:** BigQuery Conversational Analytics Prompt & Response Logging emits 9 standard event types (`INVOCATION_STARTING`, `USER_MESSAGE_RECEIVED`, `AGENT_RESPONSE`, `INVOCATION_COMPLETED`, `LLM_RESPONSE`, `EMBEDDING_SUGGESTION`, `INVOCATION_ERROR`, `AGENT_ERROR`, `LLM_ERROR`) and omits low-level ADK tool/agent spans (`AGENT_STARTING`, `AGENT_COMPLETED`, `LLM_REQUEST`, `TOOL_STARTING`, `TOOL_COMPLETED`, `TOOL_ERROR`). Everything in this notebook is designed specifically for that 9-event contract.

> **Data governance & privacy note:** Raw user prompts (`user_prompt`) and agent answers (`agent_response`) may contain sensitive business questions or PII. Keep `v_bqca_customer_sdk_events`, `v_bqca_customer_turns`, and `bqca_golden_qa` in the same BigQuery dataset as `agent_events` so dataset-level IAM policies and VPC Service Controls perimeter rules apply uniformly.

## 1. Customer configuration, compatibility views and `client.doctor()`

### 1.0 Install & authenticate

Run the first cell to install `bigquery-agent-analytics[llm,bigframes]==0.5.4` if it is not already installed in your Python kernel (Google Colab, BigQuery Studio, Vertex AI Colab Enterprise, Vertex AI Workbench, or local JupyterLab) and authenticate when inside Google Colab.

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

#@title 1.0 Install & Authenticate
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("bigquery_agent_analytics") is None:
    def _pip(*args):
        return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args]).returncode

    if _pip("bigquery-agent-analytics[llm,bigframes]==0.5.4") != 0:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "bigquery-agent-analytics[llm,bigframes] @ git+https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK.git@v0.5.4",
            ],
            check=True,
        )
    print("Installed bigquery-agent-analytics 0.5.4. If prompted to restart the runtime, restart and re-run from the top.")
else:
    print("bigquery-agent-analytics is already installed in this runtime.")

if "google.colab" in sys.modules:
    try:
        from google.colab import auth

        auth.authenticate_user()
        print("Authenticated Google Colab user.")
    except Exception as exc:
        print(f"Colab interactive auth skipped (using runtime credentials): {exc}")
else:
    print("Using active runtime Application Default Credentials (ADC).")

### 1.1 Set your project, dataset, and table parameters

Update `PROJECT_ID`, `DATASET_ID`, and `VERTEX_CONNECTION_ID` below (or set `BQCA_PROJECT_ID`, `BQCA_DATASET_ID`, `BQCA_RAW_TABLE_ID`, `BQCA_VERTEX_CONNECTION_ID` as environment variables for headless scheduled runs). Leave `RAW_TABLE_ID = "agent_events"` when running on the default table created by BQCA Prompt & Response Logging. By default, this notebook uses **`gemini-3.5-flash`** as the BigQuery ML `AI.GENERATE` / `AI.CLASSIFY` and direct Vertex AI judge endpoint.

In [ ]:
#@title 1.1 Customer Configuration Parameters
import asyncio
from importlib.metadata import version as _dist_version
import json
import logging
import os
import re
import time
import warnings

warnings.filterwarnings("ignore")
for _noisy in ("google_genai", "google_genai.models", "google.genai", "absl", "urllib3", "httpx"):
    logging.getLogger(_noisy).setLevel(logging.ERROR)
_bqaa_logger = logging.getLogger("bigquery_agent_analytics")
_bqaa_logger.handlers = [logging.StreamHandler(sys.stdout)]
_bqaa_logger.setLevel(logging.WARNING)
_bqaa_logger.propagate = False

# ---- Customer Configuration Parameters (edit in Colab form, inline, or via BQCA_* env vars) --------------------
PROJECT_ID = "your-gcp-project-id"  # @param {type:"string"}
DATASET_ID = "your_bqca_logs_dataset"  # @param {type:"string"}
RAW_TABLE_ID = "agent_events"  # @param {type:"string"}
COMPAT_VIEW_ID = "v_bqca_customer_sdk_events"  # @param {type:"string"}
TURNS_VIEW_ID = "v_bqca_customer_turns"  # @param {type:"string"}
GOLDEN_QA_TABLE_ID = "bqca_golden_qa"  # @param {type:"string"}
LOCATION = "US"  # @param {type:"string"}
VERTEX_LOCATION = "us-central1"  # @param {type:"string"}
VERTEX_CONNECTION_ID = "projects/your-gcp-project-id/locations/us/connections/bqca_vertex_connection"  # @param {type:"string"}
JUDGE_MODEL = "gemini-3.5-flash"  # @param {type:"string"}
CATEGORICAL_MODEL = "gemini-3.5-flash"  # @param {type:"string"}
EMBEDDING_MODEL = "text-embedding-005"  # @param {type:"string"}
DIRECT_JUDGE_MODEL = "gemini-3.5-flash"  # @param {type:"string"}
JUDGE_THRESHOLD = 0.6  # @param {type:"number"}
DRIFT_THRESHOLD = 0.15  # @param {type:"number"}
EVAL_LOOKBACK_DAYS = 90  # @param {type:"integer"}
MAX_EVAL_SESSIONS = 100  # @param {type:"integer"}

# Support environment variable overrides for headless CI/CD or Cloud Run Jobs:
PROJECT_ID = os.environ.get("BQCA_PROJECT_ID", PROJECT_ID)
DATASET_ID = os.environ.get("BQCA_DATASET_ID", DATASET_ID)
RAW_TABLE_ID = os.environ.get("BQCA_RAW_TABLE_ID", RAW_TABLE_ID)
LOCATION = os.environ.get("BQCA_LOCATION", LOCATION)
VERTEX_LOCATION = os.environ.get("BQCA_VERTEX_LOCATION", VERTEX_LOCATION)
VERTEX_CONNECTION_ID = os.environ.get("BQCA_VERTEX_CONNECTION_ID", VERTEX_CONNECTION_ID)

# Backward-compatible aliases used in manual snippets:
BQ_LOCATION = LOCATION
SDK_VIEW_ID = COMPAT_VIEW_ID
GEMINI_ENDPOINT = JUDGE_MODEL
CATEGORICAL_ENDPOINT = CATEGORICAL_MODEL
EMBEDDING_ENDPOINT = EMBEDDING_MODEL

if (
    PROJECT_ID == "your-gcp-project-id"
    or DATASET_ID == "your_bqca_logs_dataset"
    or "your-gcp-project-id" in VERTEX_CONNECTION_ID
):
    raise ValueError(
        "Stop! Please update PROJECT_ID, DATASET_ID, and VERTEX_CONNECTION_ID in Cell 1.1 "
        "(or set BQCA_PROJECT_ID, BQCA_DATASET_ID, and BQCA_VERTEX_CONNECTION_ID) before running the notebook. "
        "See Section 2 of docs/guides/bqca-prompt-response-logging-manual.md for setup commands."
    )

# Configure google-genai for direct Vertex AI judge calls (GraderPipeline and TrialRunner).
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = VERTEX_LOCATION

from google.api_core import exceptions as gcp_exceptions
from google.cloud import bigquery
from IPython.display import Markdown, display
import pandas as pd

import bigquery_agent_analytics as bqaa
from bigquery_agent_analytics import (
    CategoricalEvaluationConfig,
    CategoricalMetricCategory,
    CategoricalMetricDefinition,
    Client,
    GraderPipeline,
    LLMAsJudge,
    PerformanceEvaluator,
    SystemEvaluator,
    TraceFilter,
    TrialRunner,
)
from bigquery_agent_analytics.aggregate_grader import WeightedStrategy
from bigquery_agent_analytics.system_evaluator import SESSION_SUMMARY_QUERY

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

# The 9 standard event types emitted by BigQuery Conversational Analytics Prompt & Response Logging.
BQCA_EVENT_TYPES = [
    "INVOCATION_STARTING",
    "USER_MESSAGE_RECEIVED",
    "AGENT_RESPONSE",
    "INVOCATION_COMPLETED",
    "LLM_RESPONSE",
    "EMBEDDING_SUGGESTION",
    "INVOCATION_ERROR",
    "AGENT_ERROR",
    "LLM_ERROR",
]
BQCA_EVENT_ALLOWLIST = BQCA_EVENT_TYPES

print(f"bigquery-agent-analytics {_dist_version('bigquery-agent-analytics')} | pandas {pd.__version__}")
print(f"Project={PROJECT_ID} | Dataset={DATASET_ID} | BQ Location={LOCATION} | Vertex Location={VERTEX_LOCATION}")
print(f"Source table={RAW_TABLE_ID} -> Compatibility view={COMPAT_VIEW_ID} | Turns view={TURNS_VIEW_ID}")
print(f"Golden Q&A table={GOLDEN_QA_TABLE_ID} | Judge/BQML AI endpoint={JUDGE_MODEL} | Embeddings={EMBEDDING_MODEL}")

In [ ]:
#@title 1.2 Shared BigQuery Helpers & Pre-Flight Check
_IDENTIFIER_RE = re.compile(r"^[A-Za-z0-9_\-]+$")
for _name, _val in [
    ("PROJECT_ID", PROJECT_ID),
    ("DATASET_ID", DATASET_ID),
    ("RAW_TABLE_ID", RAW_TABLE_ID),
    ("COMPAT_VIEW_ID", COMPAT_VIEW_ID),
    ("TURNS_VIEW_ID", TURNS_VIEW_ID),
    ("GOLDEN_QA_TABLE_ID", GOLDEN_QA_TABLE_ID),
]:
    if not _IDENTIFIER_RE.match(_val):
        raise ValueError(f"Invalid BigQuery identifier for {_name}: {_val!r}")

bq = bqaa.make_bq_client(PROJECT_ID, location=LOCATION)
RESULTS = {}
PERSONA_BY_SESSION = {}

_TRANSIENT_BQ_EXCEPTIONS = (
    gcp_exceptions.ServiceUnavailable,
    gcp_exceptions.TooManyRequests,
    gcp_exceptions.InternalServerError,
)


def T(table_id):
    """Returns a back-ticked fully qualified BigQuery table or view identifier."""
    if not _IDENTIFIER_RE.match(table_id):
        raise ValueError(f"Invalid table_id: {table_id!r}")
    return f"`{PROJECT_ID}.{DATASET_ID}.{table_id}`"


def run_sql(sql, params=None, retries=2):
    """Executes a BigQuery SQL query and returns a pandas DataFrame with retry on transient 429/500/503 errors."""
    kinds = {bool: "BOOL", int: "INT64", float: "FLOAT64", str: "STRING"}
    query_params = [bigquery.ScalarQueryParameter(k, kinds[type(v)], v) for k, v in (params or {}).items()]
    job_config = bigquery.QueryJobConfig(query_parameters=query_params)
    for attempt in range(retries + 1):
        try:
            return bq.query(sql, job_config=job_config).to_dataframe()
        except _TRANSIENT_BQ_EXCEPTIONS:
            if attempt == retries:
                raise
            time.sleep(5 * (attempt + 1))


def md_table(df, max_chars=160):
    """Formats a pandas DataFrame as a Markdown table with truncated cells."""
    def cell(v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return ""
        s = str(v).replace("|", "\\|").replace("\n", " ")
        return s if len(s) <= max_chars else s[: max_chars - 1] + "…"

    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    lines += ["| " + " | ".join(cell(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


def show(text):
    display(Markdown(text))


def short(text, n=140):
    s = " ".join(str(text or "").split())
    return s if len(s) <= n else s[: n - 1] + "…"


def fmt(x, spec=".2f"):
    return "n/a" if x is None or pd.isna(x) else format(x, spec)


class timed:
    """Context manager that prints elapsed wall time for a step."""
    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        print(f"[{self.label}] {time.time() - self.t0:.1f}s")


def score_frame(report):
    """Converts an SDK EvaluationReport into a per-session DataFrame."""
    rows = []
    for s in report.session_scores:
        row = {"session_id": s.session_id, "passed": bool(s.passed)}
        row.update({k: (None if v is None else round(float(v), 3)) for k, v in s.scores.items()})
        row["feedback"] = (s.llm_feedback or "").replace("\n", " ")
        rows.append(row)
    if not rows:
        return pd.DataFrame(columns=["session_id", "persona", "passed", "feedback"])
    out = pd.DataFrame(rows).sort_values("session_id").reset_index(drop=True)
    if PERSONA_BY_SESSION:
        out.insert(1, "persona", out["session_id"].map(PERSONA_BY_SESSION).fillna("unattributed"))
    return out


def cat_frame(report):
    """Converts an SDK CategoricalEvaluationReport into a per-session DataFrame."""
    rows = []
    for sr in report.session_results:
        for m in sr.metrics:
            rows.append({
                "session_id": sr.session_id,
                "metric": m.metric_name,
                "category": m.category,
                "valid": m.passed_validation,
                "justification": (m.justification or "").replace("\n", " "),
            })
    if not rows:
        return pd.DataFrame(columns=["session_id", "persona", "metric", "category", "valid", "justification"])
    out = pd.DataFrame(rows).sort_values("session_id").reset_index(drop=True)
    if PERSONA_BY_SESSION:
        out.insert(1, "persona", out["session_id"].map(PERSONA_BY_SESSION).fillna("unattributed"))
    return out


print("Notebook helpers loaded:", ", ".join(["T", "run_sql", "md_table", "show", "short", "fmt", "timed", "score_frame", "cat_frame"]))

### 1.3 Provision the two customer views (`v_bqca_customer_sdk_events` and `v_bqca_customer_turns`)

Why create these two lightweight views directly over your raw `agent_events` table before running analytics?

1. **SDK Compatibility View (`v_bqca_customer_sdk_events`)**: BQCA logs `AGENT_RESPONSE.content.response` as a structured JSON object (`{"parts": [{"markdown": "..."}]}` or `{"clarifying_question": "..."}`), whereas `LLMAsJudge` in `bigquery-agent-analytics` v0.5.4 reads `JSON_VALUE(content, '$.response')` (which returns `NULL` on a JSON object unless flattened to a string). This view flattens `$.response` and `$.text_summary` to the scalar answer text while preserving the original nested object under `$.raw_response`, so every SDK method (`LLMAsJudge`, `evaluate_categorical`, `SystemEvaluator`, `drift_detection`) works out of the box.
2. **Conversation Turns View (`v_bqca_customer_turns`)**: Analysts and BI dashboards usually want **one row per session** pairing `user_prompt`, `agent_response`, `clarifying_question`, `is_fast_path`, `e2e_latency_ms`, token counts (`prompt_tokens`, `candidate_tokens`, `thinking_tokens`, `cached_tokens`, `total_tokens`), estimated token costs (`cost_sdk_usd`, `cost_incl_thinking_usd`), `finish_reasons`, and `error_messages`. Defining both views directly over `RAW_TABLE_ID` (scoped by `EVAL_LOOKBACK_DAYS`) avoids nested view-over-view overhead when running `AI.GENERATE` or `AI.EMBED`.

The next cell creates or updates both views in your dataset.

In [ ]:
#@title 1.3 Provision Customer Views (v_bqca_customer_sdk_events & v_bqca_customer_turns)
event_types_sql = ", ".join(f"'{t}'" for t in BQCA_EVENT_TYPES)
lookback_days = int(EVAL_LOOKBACK_DAYS)

CREATE_SDK_VIEW_SQL = f"""
CREATE OR REPLACE VIEW {T(COMPAT_VIEW_ID)} AS
WITH filtered AS (
  SELECT
    *,
    IF(
      event_type = 'AGENT_RESPONSE' AND content IS NOT NULL,
      COALESCE(
        NULLIF(JSON_VALUE(content, '$.text_summary'), ''),
        NULLIF(ARRAY_TO_STRING(ARRAY(
          SELECT JSON_VALUE(part, '$.markdown')
          FROM UNNEST(JSON_QUERY_ARRAY(content, '$.response.parts')) AS part
          WHERE JSON_VALUE(part, '$.markdown') IS NOT NULL
        ), '\\n\\n'), ''),
        NULLIF(JSON_VALUE(content, '$.response.clarifying_question'), ''),
        JSON_VALUE(content, '$.response')
      ),
      NULL
    ) AS answer_text
  FROM {T(RAW_TABLE_ID)}
  WHERE event_type IN ({event_types_sql})
    AND timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {lookback_days} DAY)
)
SELECT
  timestamp,
  event_id,
  event_type,
  agent,
  user_id,
  session_id,
  invocation_id,
  trace_id,
  span_id,
  parent_span_id,
  CASE
    WHEN event_type = 'AGENT_RESPONSE' AND content IS NOT NULL THEN
      JSON_SET(
        content,
        '$.raw_response', JSON_QUERY(content, '$.response'),
        '$.text_summary', answer_text,
        '$.response', answer_text
      )
    ELSE content
  END AS content,
  content_parts,
  attributes,
  latency_ms,
  status,
  error_message,
  is_truncated
FROM filtered
"""

CREATE_TURNS_VIEW_SQL = f"""
CREATE OR REPLACE VIEW {T(TURNS_VIEW_ID)} AS
SELECT
  session_id,
  ANY_VALUE(invocation_id) AS invocation_id,
  ANY_VALUE(user_id) AS user_id,
  COALESCE(
    NULLIF(MAX(JSON_VALUE(attributes, '$.session_metadata.state.custom_labels.persona')), ''),
    NULLIF(SPLIT(ANY_VALUE(user_id), '@')[SAFE_OFFSET(0)], ''),
    NULLIF(MAX(JSON_VALUE(attributes, '$.session_metadata.state."data-agent-id"')), ''),
    'unattributed'
  ) AS persona,
  MAX(JSON_VALUE(attributes, '$.session_metadata.state.custom_labels.persona_role')) AS persona_role,
  MAX(JSON_VALUE(attributes, '$.session_metadata.state.custom_labels.scenario_tag')) AS scenario_tag,
  MAX(JSON_VALUE(attributes, '$.session_metadata.state."data-agent-id"')) AS data_agent_id,
  MAX(JSON_VALUE(attributes, '$.session_metadata.state."conversation-id"')) AS conversation_id,
  MIN(timestamp) AS started_at,
  COUNT(*) AS event_count,
  COUNTIF(event_type = 'USER_MESSAGE_RECEIVED') AS turns,
  ARRAY_AGG(
    IF(
      event_type = 'USER_MESSAGE_RECEIVED',
      COALESCE(NULLIF(JSON_VALUE(content, '$.text_summary'), ''), JSON_VALUE(content, '$.prompt')),
      NULL
    )
    IGNORE NULLS ORDER BY timestamp ASC LIMIT 1
  )[SAFE_OFFSET(0)] AS user_prompt,
  ARRAY_AGG(
    IF(
      event_type = 'AGENT_RESPONSE' AND content IS NOT NULL,
      COALESCE(
        NULLIF(JSON_VALUE(content, '$.text_summary'), ''),
        NULLIF(ARRAY_TO_STRING(ARRAY(
          SELECT JSON_VALUE(part, '$.markdown')
          FROM UNNEST(JSON_QUERY_ARRAY(content, '$.response.parts')) AS part
          WHERE JSON_VALUE(part, '$.markdown') IS NOT NULL
        ), '\\n\\n'), ''),
        NULLIF(JSON_VALUE(content, '$.response.clarifying_question'), ''),
        JSON_VALUE(content, '$.response')
      ),
      NULL
    )
    IGNORE NULLS ORDER BY timestamp DESC LIMIT 1
  )[SAFE_OFFSET(0)] AS agent_response,
  MAX(IF(
    event_type = 'AGENT_RESPONSE',
    JSON_VALUE(content, '$.response.clarifying_question'),
    NULL
  )) AS clarifying_question,
  LOGICAL_OR(COALESCE(JSON_VALUE(attributes, '$.fast_path') = 'true', FALSE)) AS is_fast_path,
  MAX(IF(event_type = 'INVOCATION_COMPLETED', SAFE_CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64), NULL)) AS e2e_latency_ms,
  AVG(SAFE_CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64)) AS avg_event_latency_ms,
  AVG(SAFE_CAST(JSON_VALUE(latency_ms, '$.time_to_first_token_ms') AS FLOAT64)) AS avg_ttft_ms,
  COUNTIF(event_type = 'LLM_RESPONSE') AS llm_calls,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.prompt_token_count') AS INT64), 0)), 0) AS prompt_tokens,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.candidates_token_count') AS INT64), 0)), 0) AS candidate_tokens,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.thoughts_token_count') AS INT64), 0)), 0) AS thinking_tokens,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.cached_content_token_count') AS INT64), 0)), 0) AS cached_tokens,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.total_token_count') AS INT64), 0)), 0) AS total_tokens,
  ROUND(
    COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.prompt_token_count') AS INT64), 0)), 0) / 1000.0 * 0.00125
    + COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.candidates_token_count') AS INT64), 0)), 0) / 1000.0 * 0.01,
    6
  ) AS cost_sdk_usd,
  ROUND(
    COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.prompt_token_count') AS INT64), 0)), 0) / 1000.0 * 0.00125
    + (
      COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.candidates_token_count') AS INT64), 0)), 0)
      + COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.thoughts_token_count') AS INT64), 0)), 0)
    ) / 1000.0 * 0.01,
    6
  ) AS cost_incl_thinking_usd,
  STRING_AGG(DISTINCT IF(event_type = 'LLM_RESPONSE', COALESCE(JSON_VALUE(attributes, '$.finish_reason'), 'STOP'), NULL), ', ') AS finish_reasons,
  COALESCE(LOGICAL_OR(ENDS_WITH(event_type, '_ERROR') OR error_message IS NOT NULL OR status = 'ERROR'), FALSE) AS has_error,
  STRING_AGG(DISTINCT error_message, ' | ') AS error_messages
FROM {T(RAW_TABLE_ID)}
WHERE event_type IN ({event_types_sql})
  AND timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {lookback_days} DAY)
GROUP BY session_id
"""

bq.query(CREATE_SDK_VIEW_SQL).result()
bq.query(CREATE_TURNS_VIEW_SQL).result()
print(f"Provisioned SDK compatibility view: {T(COMPAT_VIEW_ID)} (lookback={lookback_days}d)")
print(f"Provisioned flat conversation turns view: {T(TURNS_VIEW_ID)} (lookback={lookback_days}d)")

verify_norm_df = run_sql(f"""
SELECT
  (SELECT COUNT(*) FROM {T(RAW_TABLE_ID)}) AS raw_rows,
  COUNT(*) AS sdk_view_rows,
  COUNT(DISTINCT session_id) AS distinct_sessions,
  COUNTIF(event_type = 'AGENT_RESPONSE') AS agent_response_rows,
  COUNTIF(event_type = 'AGENT_RESPONSE' AND JSON_VALUE(content, '$.response') IS NOT NULL) AS scalar_response_ok,
  COUNTIF(event_type = 'AGENT_RESPONSE' AND JSON_VALUE(content, '$.text_summary') IS NOT NULL) AS text_summary_ok,
  COUNTIF(event_type = 'AGENT_RESPONSE' AND JSON_QUERY(content, '$.raw_response') IS NOT NULL) AS raw_response_preserved
FROM {T(COMPAT_VIEW_ID)}
""")
display(verify_norm_df)

### 1.4 Initialize the SDK `Client`, run `client.doctor()` & ping Vertex AI connection

`client.doctor()` verifies the 17-column table schema, checks the BigQuery remote connection configuration for `AI.GENERATE` / `AI.CLASSIFY`, and reports event counts by `event_type`. Because BQCA intentionally omits low-level ADK tool and agent spans (`AGENT_STARTING`, `AGENT_COMPLETED`, `LLM_REQUEST`, `TOOL_STARTING`, `TOOL_COMPLETED`), `doctor()` will list those non-BQCA event types under `warnings` — that is expected on production BQCA logs. We also run a 1-row `AI.GENERATE` ping against `JUDGE_MODEL` (`gemini-3.5-flash`) to verify your Cloud Resource Connection IAM immediately.

In [ ]:
#@title 1.4 Run client.doctor() & 1-Row Vertex AI Connection Ping
client = Client(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=COMPAT_VIEW_ID,
    location=LOCATION,
    bq_client=bq,
    endpoint=JUDGE_MODEL,
    connection_id=VERTEX_CONNECTION_ID,
)

doc = client.doctor()
cols = doc["schema"].get("columns") or doc["schema"].get("present", [])
n_cols = len(cols)
cov = doc.get("event_coverage", {})
if "error" in cov:
    raise RuntimeError(f"client.doctor() failed to query {COMPAT_VIEW_ID}: {cov['error']}")
total_events = sum(v for k, v in cov.items() if isinstance(v, (int, float)))
print(f"Schema status: {doc['schema']['status']} ({n_cols} columns) | Total events in {COMPAT_VIEW_ID}: {total_events:,}")
print(f"AI.GENERATE endpoint: {doc['ai_generate']['endpoint']} | Connection configured: {bool(doc['ai_generate']['connection_id'])}")
for w in doc.get("warnings", []):
    if w.startswith("No events for types:"):
        print(f"  Expected on BQCA logs (non-BQCA ADK event types): {w}")
    else:
        print(f"  WARNING: {w}")

# Live 1-row AI.GENERATE connection & IAM ping
ping_df = run_sql(
    f"""
    SELECT (AI.GENERATE(
      prompt => 'Reply with OK if Vertex AI connection is healthy.',
      connection_id => '{VERTEX_CONNECTION_ID}',
      endpoint => '{JUDGE_MODEL}',
      model_params => JSON '{{"generationConfig": {{"temperature": 0.0, "maxOutputTokens": 64}}}}',
      output_schema => 'verdict STRING'
    )).verdict AS ai_generate_ping
    """,
    retries=0,
)
print(f"Live 1-row AI.GENERATE ping ({JUDGE_MODEL}): {ping_df.ai_generate_ping.iloc[0]!r}")

all_types = sorted(set(BQCA_EVENT_TYPES) | set(cov))
cov_df = pd.DataFrame({
    "event_type": all_types,
    "in_bqca_event_contract": ["yes" if t in BQCA_EVENT_TYPES else "no" for t in all_types],
    "event_count": [cov.get(t, 0) for t in all_types],
}).sort_values(["in_bqca_event_contract", "event_count"], ascending=[False, False]).reset_index(drop=True)
display(cov_df)

RESULTS["s1"] = {
    "schema_status": doc["schema"]["status"],
    "columns": n_cols,
    "total_events": int(total_events),
    "active_event_types": sum(1 for v in cov.values() if isinstance(v, (int, float)) and v > 0),
    "agent_response_rows": int(verify_norm_df.agent_response_rows.iloc[0]),
    "scalar_responses": int(verify_norm_df.scalar_response_ok.iloc[0]),
}
show(f"""**Section 1 Takeaway (live)**

- `client.doctor()` confirmed schema `{RESULTS['s1']['schema_status']}` (**{RESULTS['s1']['columns']}** columns) with **{RESULTS['s1']['total_events']}** events across **{RESULTS['s1']['active_event_types']}** active BQCA event types, and the 1-row `AI.GENERATE` ping on `{JUDGE_MODEL}` succeeded.
- In `{COMPAT_VIEW_ID}`, **{RESULTS['s1']['scalar_responses']}/{RESULTS['s1']['agent_response_rows']}** `AGENT_RESPONSE` rows expose both scalar `$.response` and `$.text_summary` while keeping the original nested payload in `$.raw_response`.""")

## 2. Prompt & Response Conversation Explorer (`v_bqca_customer_turns`)

Before running LLM judges, inspect `v_bqca_customer_turns` to see what your users are asking, how your agents are responding, how often the agent asks a clarifying question (`ask_clarifying_question`), and which turns hit the Verified Queries fast path (`is_fast_path = TRUE`).

In [ ]:
#@title 2.1 Conversation Explorer & Cohort Summary
turns_df = run_sql(f"SELECT * FROM {T(TURNS_VIEW_ID)} ORDER BY persona, session_id")
if turns_df.empty:
    raise RuntimeError(
        f"No conversation turns found in {T(TURNS_VIEW_ID)} within the last {EVAL_LOOKBACK_DAYS} days. "
        f"Verify that {T(RAW_TABLE_ID)} contains BQCA events or increase EVAL_LOOKBACK_DAYS in Cell 1.1."
    )
PERSONA_BY_SESSION.update(dict(zip(turns_df["session_id"], turns_df["persona"])))

cohort_summary_df = (
    turns_df.groupby(["data_agent_id", "persona", "persona_role"], dropna=False)
    .agg(
        sessions=("session_id", "count"),
        answered_turns=("agent_response", lambda s: int(s.notna().sum())),
        clarifying_turns=("clarifying_question", lambda s: int(s.notna().sum())),
        fast_path_turns=("is_fast_path", lambda s: int(s.astype(bool).sum())),
        error_turns=("has_error", lambda s: int(s.astype(bool).sum())),
        median_e2e_latency_ms=("e2e_latency_ms", "median"),
        total_tokens=("total_tokens", "sum"),
    )
    .round({"median_e2e_latency_ms": 0})
    .reset_index()
)
print("Conversation summary by data_agent_id and user cohort:")
display(cohort_summary_df)

sample_turns_df = (
    turns_df.groupby("persona").head(1)[
        ["persona", "session_id", "is_fast_path", "has_error", "e2e_latency_ms", "total_tokens", "user_prompt", "agent_response"]
    ].reset_index(drop=True)
)
show("**Sample paired `user_prompt` -> `agent_response` per user cohort**\n\n" + md_table(sample_turns_df, max_chars=150))

clarify_df = turns_df[turns_df.clarifying_question.notna()][
    ["persona", "session_id", "user_prompt", "clarifying_question"]
].reset_index(drop=True)
if not clarify_df.empty:
    print("Turns where the agent asked a clarifying question (extracted from $.raw_response.clarifying_question):")
    display(clarify_df)

RESULTS["s2"] = {
    "sessions": len(turns_df),
    "cohorts": int(turns_df.persona.nunique()),
    "answered": int(turns_df.agent_response.notna().sum()),
    "clarifying": int(turns_df.clarifying_question.notna().sum()),
    "fast_path": int(turns_df.is_fast_path.astype(bool).sum()),
    "errors": int(turns_df.has_error.astype(bool).sum()),
}
show(f"""**Section 2 Takeaway (live)**

- `{TURNS_VIEW_ID}` pairs **{RESULTS['s2']['sessions']}** sessions across **{RESULTS['s2']['cohorts']}** user cohorts: **{RESULTS['s2']['answered']}** delivered an `AGENT_RESPONSE` (including **{RESULTS['s2']['clarifying']}** clarifying-question turn), **{RESULTS['s2']['fast_path']}** used the Verified Queries fast path, and **{RESULTS['s2']['errors']}** ended with a sanitised error event.""")

## 3. Customer sentiment & frustrated-conversation triage

Two complementary SDK methods let you monitor user satisfaction directly inside BigQuery (`AI.GENERATE` / `AI.CLASSIFY`):

- **3.1 `LLMAsJudge.sentiment()`**: Scores overall interaction satisfaction and helpfulness on `[0.0, 1.0]`. Joining those scores to `v_bqca_customer_turns` gives your support and product teams an immediate triage queue of low-scoring conversations with the exact prompt, response, and judge explanation.
- **3.2 `evaluate_categorical()` Tone Rubric**: Classifies each conversation into your own UX buckets (`Positive_Satisfied`, `Neutral_Analytical`, `Clarification_Needed`, `Frustrated_Impatient`) with a written justification.

In [ ]:
#@title 3.1 Sentiment Evaluation (LLMAsJudge.sentiment)
with timed(f"LLMAsJudge.sentiment via client.evaluate ({JUDGE_MODEL})"):
    sentiment_report = client.evaluate(
        LLMAsJudge.sentiment(threshold=JUDGE_THRESHOLD, model=JUDGE_MODEL),
        filters=TraceFilter(limit=MAX_EVAL_SESSIONS),
    )
print(sentiment_report.summary())
print("Execution details:", {k: sentiment_report.details.get(k) for k in ("execution_mode", "fallback_reason", "parse_errors") if sentiment_report.details.get(k) is not None})
sent_df = score_frame(sentiment_report)

# Join sentiment scores with v_bqca_customer_turns to inspect the lowest-scoring conversations.
sent_triage_df = (
    sent_df.merge(
        turns_df[["session_id", "user_prompt", "agent_response", "has_error", "finish_reasons"]],
        on="session_id",
        how="left",
    )
    .sort_values(["sentiment", "session_id"], na_position="first")
    .reset_index(drop=True)
)
print(f"Triage queue — sessions that failed the {JUDGE_THRESHOLD} sentiment threshold (or returned no score):")
display(
    sent_triage_df[~sent_triage_df.passed][
        ["persona", "session_id", "sentiment", "has_error", "finish_reasons", "user_prompt", "feedback"]
    ]
)

sent_by_cohort_df = (
    sent_df.groupby("persona")
    .agg(
        sessions=("session_id", "count"),
        passed=("passed", lambda s: int(s.sum())),
        failed=("passed", lambda s: int((~s).sum())),
        mean_sentiment=("sentiment", "mean"),
        min_sentiment=("sentiment", "min"),
    )
    .round(2)
    .reset_index()
)
print("Sentiment summary by user cohort:")
display(sent_by_cohort_df)

In [ ]:
#@title 3.2 UX & Tone Categorical Rubric (evaluate_categorical)
ux_tone_metric = CategoricalMetricDefinition(
    name="customer_ux_tone",
    definition="Overall user experience and tone across the session, judged from the user's prompt and how well the agent addressed their need.",
    categories=[
        CategoricalMetricCategory(
            name="Positive_Satisfied",
            definition="The user is appreciative or satisfied, and the agent clearly answered their question.",
        ),
        CategoricalMetricCategory(
            name="Neutral_Analytical",
            definition="Matter-of-fact analytical question and response with no strong emotional signal.",
        ),
        CategoricalMetricCategory(
            name="Clarification_Needed",
            definition="The user's request was underspecified or ambiguous, and the agent asked a clarifying question.",
        ),
        CategoricalMetricCategory(
            name="Frustrated_Impatient",
            definition="The user expresses frustration, impatience, or dissatisfaction with the agent or its output.",
        ),
    ],
)
ux_tone_cfg = CategoricalEvaluationConfig(
    metrics=[ux_tone_metric],
    endpoint=CATEGORICAL_MODEL,
    connection_id=VERTEX_CONNECTION_ID,
    include_justification=True,
)
with timed(f"evaluate_categorical: customer_ux_tone ({CATEGORICAL_MODEL})"):
    ux_tone_report = client.evaluate_categorical(ux_tone_cfg, filters=TraceFilter(limit=MAX_EVAL_SESSIONS))
print(ux_tone_report.summary())
print("Execution mode:", ux_tone_report.details.get("execution_mode"))
ux_tone_df = cat_frame(ux_tone_report)
if not ux_tone_df.empty:
    print("User tone distribution by cohort:")
    display(pd.crosstab(ux_tone_df["persona"], ux_tone_df["category"]))

flagged_ux_df = ux_tone_df[ux_tone_df.category.isin(["Frustrated_Impatient", "Clarification_Needed"])][
    ["persona", "session_id", "category", "justification"]
].reset_index(drop=True)
print("Sessions flagged as Frustrated_Impatient or Clarification_Needed:")
display(flagged_ux_df)

RESULTS["s3"] = {
    "sessions": len(sent_df),
    "passed": int(sent_df.passed.sum()),
    "failed": int((~sent_df.passed).sum()),
    "tone_counts": ux_tone_df.category.value_counts().to_dict(),
    "frustrated_sessions": ux_tone_df[ux_tone_df.category == "Frustrated_Impatient"].session_id.tolist(),
    "clarification_sessions": ux_tone_df[ux_tone_df.category == "Clarification_Needed"].session_id.tolist(),
}
show(f"""**Section 3 Takeaway (live)**

- **Sentiment judge (`LLMAsJudge.sentiment`)**: **{RESULTS['s3']['passed']}/{RESULTS['s3']['sessions']}** sessions passed the `{JUDGE_THRESHOLD}` threshold (**{RESULTS['s3']['passed'] / max(RESULTS['s3']['sessions'], 1):.0%}**); **{RESULTS['s3']['failed']}** sessions were routed to the triage table above.
- **UX & Tone rubric (`evaluate_categorical`)**: {', '.join(f"**{k}** {v}" for k, v in RESULTS['s3']['tone_counts'].items())}. Frustrated session(s): {', '.join(f"`{s}`" for s in RESULTS['s3']['frustrated_sessions']) or 'none'}; clarification-needed session(s): {', '.join(f"`{s}`" for s in RESULTS['s3']['clarification_sessions']) or 'none'}.""")

## 4. User question categorisation & topic taxonomy (`AI.CLASSIFY` & `AI.GENERATE`)

Understanding **what kinds of questions** users ask helps data owners prioritize new tables, column descriptions, and Verified Queries. With `client.evaluate_categorical()`, you can run your taxonomy in two modes:

- **Fast Bulk Mode (`include_justification=False`)**: Uses BigQuery `AI.CLASSIFY` for low-latency, low-cost classification over large log volumes.
- **Audited Mode (`include_justification=True`)**: Uses BigQuery `AI.GENERATE` with structured output to return both the category label and a 1-sentence explanation.

In [ ]:
#@title 4.1 Question Topic Taxonomy (AI.CLASSIFY vs AI.GENERATE)
topic_metric = CategoricalMetricDefinition(
    name="question_topic",
    definition="Primary analytical intent of the user's question in this session.",
    categories=[
        CategoricalMetricCategory(
            name="Ranking_TopN",
            definition="Top-N, bottom-N, or ranking questions (for example, top 5 schools by SAT score).",
        ),
        CategoricalMetricCategory(
            name="Aggregation_KPI",
            definition="Statewide totals, counts, averages, percentages, or headline KPIs.",
        ),
        CategoricalMetricCategory(
            name="Cohort_Comparison",
            definition="Comparing two or more counties, charter vs. non-charter schools, or demographic tiers.",
        ),
        CategoricalMetricCategory(
            name="Schema_Metadata",
            definition="Questions about available tables, join keys, column meanings, or record counts.",
        ),
        CategoricalMetricCategory(
            name="Out_of_Scope_Drift",
            definition="Questions outside the connected California schools dataset (other states, unrelated industries, or unlogged PII).",
        ),
    ],
)

topic_reports = {}
for include_justification in (False, True):
    mode_label = "AI.GENERATE (with rationale)" if include_justification else "AI.CLASSIFY (fast bulk)"
    cfg = CategoricalEvaluationConfig(
        metrics=[topic_metric],
        endpoint=CATEGORICAL_MODEL,
        connection_id=VERTEX_CONNECTION_ID,
        include_justification=include_justification,
    )
    with timed(f"evaluate_categorical: question_topic [{mode_label}] ({CATEGORICAL_MODEL})"):
        topic_reports[include_justification] = client.evaluate_categorical(cfg, filters=TraceFilter(limit=MAX_EVAL_SESSIONS))
    print(f"  -> {mode_label} execution_mode={topic_reports[include_justification].details.get('execution_mode')}")

df_classify = cat_frame(topic_reports[False])
df_generate = cat_frame(topic_reports[True])

topic_dist_df = (
    pd.DataFrame({
        "AI.CLASSIFY (fast)": df_classify.category.value_counts(),
        "AI.GENERATE (audited)": df_generate.category.value_counts(),
    })
    .fillna(0)
    .astype(int)
    .sort_values("AI.GENERATE (audited)", ascending=False)
)
print("Question topic distribution across both execution engines:")
display(topic_dist_df)

topic_merged_df = (
    df_classify[["session_id", "persona", "category"]]
    .merge(df_generate[["session_id", "category", "justification"]], on="session_id", suffixes=("_classify", "_generate"))
    .sort_values(["persona", "session_id"])
    .reset_index(drop=True)
)
mode_agreement = int((topic_merged_df.category_classify == topic_merged_df.category_generate).sum())
print(f"Engine agreement: AI.CLASSIFY and AI.GENERATE assigned the same category on {mode_agreement}/{len(topic_merged_df)} sessions.")
if not df_generate.empty:
    print("\nQuestion topics by user cohort (AI.GENERATE):")
    display(pd.crosstab(df_generate["persona"], df_generate["category"]))

RESULTS["s4"] = {
    "sessions": len(topic_merged_df),
    "agree": mode_agreement,
    "distribution": df_generate.category.value_counts().to_dict(),
    "out_of_scope": int((df_generate.category == "Out_of_Scope_Drift").sum()),
}
show(f"""**Section 4 Takeaway (live)**

- Across **{RESULTS['s4']['sessions']}** sessions, `AI.CLASSIFY` and `AI.GENERATE` agreed on **{RESULTS['s4']['agree']}/{RESULTS['s4']['sessions']}** topic labels (**{RESULTS['s4']['agree'] / max(RESULTS['s4']['sessions'], 1):.0%}**).
- Topic breakdown: {', '.join(f"**{k}** {v}" for k, v in RESULTS['s4']['distribution'].items())}. Use `AI.CLASSIFY` for daily dashboards and `AI.GENERATE` when auditing specific cohorts.""")

## 5. Semantic drift detection & Golden Question Bank coverage

### 5.1 Bootstrap or load `bqca_golden_qa` & run `client.drift_detection()`

As your users adopt Conversational Analytics, two coverage questions arise:

1. **Golden Coverage (`client.drift_detection()`)**: How much of your curated Golden Question Bank is actually being exercised in production traffic, and which golden questions have never been asked?
2. **Production Semantic Drift (`AI.EMBED` + `ML.DISTANCE`)**: Which incoming user questions are semantically far (`cosine_distance > DRIFT_THRESHOLD`) from *every* question in your Golden Question Bank — signaling new datasets or verified queries you should add?

> **How starter bootstrapping works:** If `bqca_golden_qa` does not exist yet in your dataset, Cell `5.1` runs `CREATE TABLE IF NOT EXISTS` to seed an initial template table from up to 50 clean answered turns with `verified = FALSE` and `source = 'auto_bootstrap_template'`. Auto-seeded rows allow the drift and SQL grading pipelines to run on your first click, and print a reminder showing how to replace them with SME-verified ground truth (`verified = TRUE`).

In [ ]:
#@title 5.1 Bootstrap or Load Golden Question Bank & Run client.drift_detection()
BOOTSTRAP_GOLDEN_SQL = f"""
CREATE TABLE IF NOT EXISTS {T(GOLDEN_QA_TABLE_ID)} AS
SELECT
  FORMAT('G%02d', ROW_NUMBER() OVER (ORDER BY started_at ASC, session_id ASC)) AS golden_id,
  COALESCE(NULLIF(persona_role, ''), 'Auto-Seeded Template') AS category,
  user_prompt AS golden_question,
  agent_response AS golden_answer,
  user_prompt AS question,
  agent_response AS expected_answer,
  session_id AS source_session_id,
  'auto_bootstrap_template' AS source,
  FALSE AS verified
FROM {T(TURNS_VIEW_ID)}
WHERE user_prompt IS NOT NULL
  AND agent_response IS NOT NULL
  AND NOT has_error
  AND clarifying_question IS NULL
ORDER BY started_at ASC, session_id ASC
LIMIT 50
"""
bq.query(BOOTSTRAP_GOLDEN_SQL).result()

golden_table_cols = {field.name for field in bq.get_table(f"{PROJECT_ID}.{DATASET_ID}.{GOLDEN_QA_TABLE_ID}").schema}
has_verified_col = "verified" in golden_table_cols
has_source_sid_col = "source_session_id" in golden_table_cols

golden_overview_df = run_sql(f"""
SELECT
  category,
  {"BOOL_AND(COALESCE(verified, FALSE))" if has_verified_col else "TRUE"} AS all_verified,
  COUNT(*) AS golden_questions
FROM {T(GOLDEN_QA_TABLE_ID)}
GROUP BY category
ORDER BY golden_questions DESC, category
""")
print(f"Golden Question Bank ({GOLDEN_QA_TABLE_ID}) summary by category:")
display(golden_overview_df)

if not bool(golden_overview_df["all_verified"].all()):
    print(
        "⚠️ [UNVERIFIED AUTO-SEEDED TEMPLATE] Your bqca_golden_qa table contains auto-seeded rows (verified = FALSE) "
        "bootstrapped from the agent's own responses so Sections 5 & 6A can run out-of-the-box.\n"
        "   Before using Section 6A as an accuracy KPI, verify the answers with a domain SME and run:\n"
        f"   UPDATE {T(GOLDEN_QA_TABLE_ID)} SET golden_answer = '<verified answer>', expected_answer = '<verified answer>', "
        "verified = TRUE, source = 'sme_curated' WHERE golden_id = 'G01';"
    )

with timed(f"client.drift_detection (AI.EMBED + ML.DISTANCE, {EMBEDDING_MODEL})"):
    drift_report = client.drift_detection(
        golden_dataset=GOLDEN_QA_TABLE_ID,
        filters=TraceFilter(limit=MAX_EVAL_SESSIONS),
        embedding_model=EMBEDDING_MODEL,
    )
print(drift_report.summary())
print("drift_detection method:", drift_report.details.get("method"))
SDK_DRIFT_THRESHOLD = drift_report.details.get("similarity_threshold", 0.3)

print("\nUncovered Golden Questions (never asked within cosine distance threshold):")
for q in drift_report.uncovered_questions:
    print("  -", short(q, 180))

In [ ]:
#@title 5.2 Production-Centric Semantic Drift Query (AI.EMBED + ML.DISTANCE)
verified_expr = "COALESCE(verified, FALSE)" if has_verified_col else "TRUE"
source_sid_expr = "source_session_id" if has_source_sid_col else "CAST(NULL AS STRING)"

MATCH_CTE = f"""
prod AS (
  SELECT
    session_id,
    user_id,
    persona,
    user_prompt AS prod_question,
    agent_response AS prod_answer
  FROM {T(TURNS_VIEW_ID)}
  WHERE user_prompt IS NOT NULL
  ORDER BY started_at DESC, session_id
  LIMIT {int(MAX_EVAL_SESSIONS)}
),
prod_emb AS (
  SELECT
    *,
    AI.EMBED(
      prod_question,
      connection_id => '{VERTEX_CONNECTION_ID}',
      endpoint => '{EMBEDDING_MODEL}'
    ).result AS q_emb
  FROM prod
),
gold_emb AS (
  SELECT
    golden_id,
    category AS golden_category,
    COALESCE(golden_question, question) AS golden_question,
    COALESCE(golden_answer, expected_answer) AS golden_answer,
    {verified_expr} AS golden_verified,
    {source_sid_expr} AS source_session_id,
    AI.EMBED(
      COALESCE(golden_question, question),
      connection_id => '{VERTEX_CONNECTION_ID}',
      endpoint => '{EMBEDDING_MODEL}'
    ).result AS q_emb
  FROM {T(GOLDEN_QA_TABLE_ID)}
),
matched AS (
  SELECT
    p.session_id,
    p.user_id,
    p.persona,
    p.prod_question,
    p.prod_answer,
    g.golden_id,
    g.golden_category,
    g.golden_question,
    g.golden_answer,
    g.golden_verified,
    g.source_session_id,
    ML.DISTANCE(p.q_emb, g.q_emb, 'COSINE') AS question_distance
  FROM prod_emb p
  CROSS JOIN gold_emb g
  QUALIFY ROW_NUMBER() OVER (PARTITION BY p.session_id ORDER BY question_distance ASC) = 1
)"""

with timed(f"Production-centric semantic drift query (AI.EMBED + ML.DISTANCE, {EMBEDDING_MODEL})"):
    matched_df = run_sql("WITH " + MATCH_CTE + "\nSELECT * FROM matched ORDER BY question_distance DESC, session_id")

drift_by_persona_df = (
    matched_df.groupby("persona")
    .agg(
        sessions=("session_id", "count"),
        min_distance=("question_distance", "min"),
        median_distance=("question_distance", "median"),
        max_distance=("question_distance", "max"),
        drifted_beyond_threshold=("question_distance", lambda s: int((s > DRIFT_THRESHOLD).sum())),
        drifted_beyond_sdk_030=("question_distance", lambda s: int((s > SDK_DRIFT_THRESHOLD).sum())),
    )
    .round(3)
    .sort_values("median_distance", ascending=False)
    .reset_index()
)
print(f"Semantic distance to nearest Golden Question by user cohort (DRIFT_THRESHOLD={DRIFT_THRESHOLD}):")
display(drift_by_persona_df)

drifted_prompts_df = (
    matched_df[matched_df.question_distance > DRIFT_THRESHOLD]
    .assign(
        cosine_distance=lambda d: d.question_distance.round(3),
        user_question=lambda d: d.prod_question.map(lambda q: short(q, 110)),
    )[["persona", "session_id", "cosine_distance", "golden_id", "user_question"]]
    .reset_index(drop=True)
)
print(f"Drifted / novel production questions (cosine_distance > {DRIFT_THRESHOLD}; n={len(drifted_prompts_df)}):")
display(drifted_prompts_df)

if not drift_by_persona_df.empty:
    top_drift_row = drift_by_persona_df.iloc[0]
    top_drift_persona = str(top_drift_row.persona)
    top_drift_median = float(top_drift_row.median_distance)
else:
    top_drift_persona = "n/a"
    top_drift_median = 0.0

RESULTS["s5"] = {
    "coverage_pct": float(drift_report.coverage_percentage),
    "covered_golden": len(drift_report.covered_questions),
    "total_golden": int(drift_report.total_golden),
    "uncovered_golden": len(drift_report.uncovered_questions),
    "drifted_over_015": len(drifted_prompts_df),
    "top_drift_persona": top_drift_persona,
    "top_drift_median": top_drift_median,
}
show(f"""**Section 5 Takeaway (live)**

- **Golden Question Bank coverage**: **{RESULTS['s5']['coverage_pct']:.1f}%** (**{RESULTS['s5']['covered_golden']}/{RESULTS['s5']['total_golden']}** golden questions covered within SDK cosine distance `{SDK_DRIFT_THRESHOLD}`; **{RESULTS['s5']['uncovered_golden']}** uncovered).
- **Emerging / drifted user prompts**: **{RESULTS['s5']['drifted_over_015']}** production sessions had `cosine_distance > {DRIFT_THRESHOLD}` from the nearest golden question. The most drifted cohort is `{RESULTS['s5']['top_drift_persona']}` (median cosine distance **{RESULTS['s5']['top_drift_median']:.3f}**).""")

## 6. Response-quality evaluation (Golden Q&A key grading & SDK judges)

### 6A. Automated Golden Q&A Answer Key Grading (`AI.EMBED` + `AI.GENERATE`)

Because BQCA logs the user's question (`USER_MESSAGE_RECEIVED`) and the agent's final response (`AGENT_RESPONSE`) — and omits intermediate `TOOL_*` SQL result rows — there are two complementary ways to evaluate response quality:

- **6A: Automated Golden Q&A Answer Key Grading (Recommended for accuracy audits)**: Match each in-domain production question (`question_distance <= DRIFT_THRESHOLD`) to its verified Golden Answer in `bqca_golden_qa`, and use `AI.GENERATE` (`gemini-3.5-flash`) to grade whether the agent's numbers and entities match the answer key (`PASS` / `FAIL`). When `bqca_golden_qa` contains SME-verified rows (`verified = TRUE`), this catches fluent hallucinations (such as `session_hallucinated_retrieval_demo`) with 100% precision using only the events BQCA logs.
- **6B: Reference-Free SDK Judges, `GraderPipeline`, and Multi-Trial Judge Consistency (`TrialRunner`)**: Run `LLMAsJudge.correctness()`, `LLMAsJudge.hallucination()`, a weighted `GraderPipeline`, and multi-trial `TrialRunner` judge consistency checks (`pass@3` and `pass^3`).

In [ ]:
#@title 6A. Golden Q&A Answer Key Grading (AI.EMBED + ML.DISTANCE + AI.GENERATE)
GOLDEN_MATCH_MAX_DISTANCE = DRIFT_THRESHOLD
HALLUCINATED_SESSION = "session_hallucinated_retrieval_demo"
REFERENCE_SESSION = "session_lena_top_math_01"

GENERATE_PARAMS = """JSON '{"generationConfig": {"temperature": 0.0, "maxOutputTokens": 1024}}'"""
GRADER_INSTRUCTIONS = """You are grading an analytics agent answer against a verified golden answer.
Decide whether the AGENT ANSWER is factually consistent with the GOLDEN ANSWER.
verdict = PASS when every entity and number stated in the golden answer appears in the agent answer with the same value, and the agent answer does not contradict the golden answer. Additional correct detail, different formatting or wording, and a different ordering of the same facts are allowed.
verdict = FAIL when the agent answer omits a golden entity or number, states a different value or entity for the same quantity, invents entities or numbers that conflict with the golden answer, or gives no answer.
justification = one sentence naming the first mismatching value, or confirming the match."""

GRADE_SQL = "WITH " + MATCH_CTE + f"""
SELECT
  m.persona,
  m.session_id,
  m.golden_id,
  m.golden_verified AS verified,
  COALESCE(m.session_id = m.source_session_id, FALSE) AS is_self_seeded,
  ROUND(m.question_distance, 3) AS question_distance,
  gen.verdict AS verdict,
  gen.justification AS justification
FROM (
  SELECT
    m.*,
    AI.GENERATE(
      prompt => CONCAT(
        @instructions, CHR(10), CHR(10),
        'Question: ', m.prod_question, CHR(10), CHR(10),
        'GOLDEN ANSWER (ground truth):', CHR(10), m.golden_answer, CHR(10), CHR(10),
        'AGENT ANSWER:', CHR(10), COALESCE(m.prod_answer, 'N/A')),
      endpoint => '{JUDGE_MODEL}',
      connection_id => '{VERTEX_CONNECTION_ID}',
      model_params => {GENERATE_PARAMS},
      output_schema => 'verdict STRING, justification STRING'
    ) AS gen
  FROM matched m
  WHERE m.question_distance <= @max_distance
) AS m
ORDER BY verdict ASC, persona, session_id
"""

with timed(f"Golden Q&A Answer Key grading via AI.EMBED + ML.DISTANCE + AI.GENERATE ({JUDGE_MODEL})"):
    graded_df = run_sql(GRADE_SQL, {"instructions": GRADER_INSTRUCTIONS, "max_distance": GOLDEN_MATCH_MAX_DISTANCE})
graded_df["verdict"] = graded_df["verdict"].astype("string").str.strip().str.upper()

verified_graded_df = graded_df[graded_df["verified"] & ~graded_df["is_self_seeded"]]
template_graded_df = graded_df[~graded_df["verified"] | graded_df["is_self_seeded"]]
if not template_graded_df.empty:
    print(
        f"⚠️ Note: {len(template_graded_df)}/{len(graded_df)} graded sessions matched unverified auto-seeded template rows "
        "(pipeline smoke test only). Verify rows in bqca_golden_qa (SET verified = TRUE) for ground-truth accuracy."
    )
print(
    f"Graded {len(graded_df)} in-domain sessions (question_distance <= {GOLDEN_MATCH_MAX_DISTANCE}) against {GOLDEN_QA_TABLE_ID} "
    f"(verified ground-truth matches: {len(verified_graded_df)}):"
)
display(graded_df.drop(columns=["is_self_seeded"]))

# Side-by-side inspection of any FAIL verdict
failed_grade_df = graded_df[graded_df.verdict == "FAIL"]
if not failed_grade_df.empty:
    bad_sid = failed_grade_df.session_id.iloc[0]
    bad_match = matched_df.set_index("session_id").loc[bad_sid]
    bad_grade = graded_df.set_index("session_id").loc[bad_sid]
    side_by_side_df = pd.DataFrame({
        "Stage": [f"Golden Answer ({bad_match.golden_id})", f"Logged Agent Answer ({bad_sid})", "AI.GENERATE Grader Verdict"],
        "Content": [
            short(bad_match.golden_answer, 420),
            short(bad_match.prod_answer, 420),
            f"{bad_grade.verdict}: {bad_grade.justification}",
        ],
    })
    show("**Side-by-side audit of the failing session caught by Golden Q&A grading**\n\n" + md_table(side_by_side_df, max_chars=430))

### 6B. Reference-free SDK judges (`correctness` & `hallucination`), `GraderPipeline`, and multi-trial judge consistency (`TrialRunner`)

Below we run the SDK's built-in `LLMAsJudge.correctness()` and `LLMAsJudge.hallucination()`, combine them with a latency SLO in `GraderPipeline`, and provide the drop-in `BQCAPerformanceEvaluator` adapter so `TrialRunner` reads `AGENT_RESPONSE` on BQCA logs. Because `TrialRunner` re-evaluates a logged session across `k=3` independent judge trials, `pass@3` and `pass^3` measure **multi-trial LLM judge consistency** on that trace (whether the judge verdict is stable across repeated calls).

In [ ]:
#@title 6B. Reference-Free Judges, GraderPipeline & Multi-Trial Consistency (TrialRunner)
with timed(f"LLMAsJudge.correctness & LLMAsJudge.hallucination on {COMPAT_VIEW_ID} ({JUDGE_MODEL})"):
    corr_report = client.evaluate(
        LLMAsJudge.correctness(threshold=JUDGE_THRESHOLD, model=JUDGE_MODEL),
        filters=TraceFilter(limit=MAX_EVAL_SESSIONS),
    )
    hall_report = client.evaluate(
        LLMAsJudge.hallucination(threshold=JUDGE_THRESHOLD, model=JUDGE_MODEL),
        filters=TraceFilter(limit=MAX_EVAL_SESSIONS),
    )

corr_df = score_frame(corr_report).set_index("session_id")
hall_df = score_frame(hall_report).set_index("session_id")
corr_pass_rate = float(corr_df["passed"].mean()) if not corr_df.empty else 0.0
hall_pass_rate = float(hall_df["passed"].mean()) if not hall_df.empty else 0.0
print(f"LLMAsJudge.correctness: {int(corr_df['passed'].sum())}/{len(corr_df)} passed ({corr_pass_rate:.0%}, mode={corr_report.details.get('execution_mode')})")
print(f"LLMAsJudge.hallucination: {int(hall_df['passed'].sum())}/{len(hall_df)} passed ({hall_pass_rate:.0%}, mode={hall_report.details.get('execution_mode')})")

judge_summary_df = pd.DataFrame({
    "persona": corr_df["persona"],
    "correctness": corr_df["correctness"],
    "correctness_passed": corr_df["passed"],
    "faithfulness": hall_df["faithfulness"],
    "faithfulness_passed": hall_df["passed"],
})

# Select target sessions for deep-dive comparison (uses the benchmark twin pair if present, else top/bottom sessions):
target_sids = [s for s in (REFERENCE_SESSION, HALLUCINATED_SESSION) if s in judge_summary_df.index]
if len(target_sids) < 2:
    target_sids = list(judge_summary_df.sort_values("correctness", ascending=False).index[:2])

print("Reference-free SDK judge summary on target comparison sessions:")
display(judge_summary_df.loc[target_sids])


class BQCAPerformanceEvaluator(PerformanceEvaluator):
    """Adapter for BQCA logs: populates SessionTrace.final_response from AGENT_RESPONSE."""

    async def get_session_trace(self, session_id, **kwargs):
        session_trace = await super().get_session_trace(session_id, **kwargs)
        for event in reversed(session_trace.events):
            if event.event_type == "AGENT_RESPONSE" and isinstance(event.content, dict):
                session_trace.final_response = (
                    event.content.get("text_summary") or event.content.get("response") or session_trace.final_response
                )
                break
        return session_trace


bqca_evaluator = BQCAPerformanceEvaluator(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=COMPAT_VIEW_ID,
    client=bq,
    llm_judge_model=DIRECT_JUDGE_MODEL,
    include_event_types=list(PerformanceEvaluator._DEFAULT_EVENT_TYPES) + ["AGENT_RESPONSE"],
)

# Run GraderPipeline (weighted latency + correctness + hallucination) and TrialRunner (3 judge consistency trials)
PIPELINE_WEIGHTS = {"latency_evaluator": 0.2, "correctness_judge": 0.5, "hallucination_judge": 0.3}
INFORMATIONAL_METRICS = [
    "response_match", "llm_judge_sentiment", "llm_judge_hallucination",
    "llm_judge_tool_usage_correct", "llm_judge_sound_reasoning", "llm_judge_efficiency",
]
TRIAL_THRESHOLDS = {m: 0.0 for m in INFORMATIONAL_METRICS}
golden_by_session = matched_df.set_index("session_id")["golden_answer"].to_dict() if not matched_df.empty else {}
graded_verdict_by_session = graded_df.set_index("session_id")["verdict"].to_dict() if not graded_df.empty else {}
runner = TrialRunner(bqca_evaluator, num_trials=3, concurrency=3)

twin_eval_rows = []
for sid in target_sids:
    try:
        summary_sql = SESSION_SUMMARY_QUERY.format(
            project=PROJECT_ID, dataset=DATASET_ID, table=COMPAT_VIEW_ID, where="session_id = @sid"
        )
        job_config = bigquery.QueryJobConfig(query_parameters=[
            bigquery.ScalarQueryParameter("sid", "STRING", sid),
            bigquery.ScalarQueryParameter("trace_limit", "INT64", 10),
        ])
        summary_rows = list(bq.query(summary_sql, job_config=job_config).result())
        if not summary_rows:
            continue
        summary = dict(summary_rows[0])
        trace = client.get_session_trace(sid)
        trace_text = "\n".join(f"{s.event_type}: {(s.content or {}).get('text_summary') or ''}" for s in trace.spans)
        answer = next((s.content.get("text_summary") for s in reversed(trace.spans) if s.event_type == "AGENT_RESPONSE" and isinstance(s.content, dict)), "")

        pipeline = GraderPipeline(WeightedStrategy(weights=PIPELINE_WEIGHTS, threshold=JUDGE_THRESHOLD))
        pipeline.add_system_grader(SystemEvaluator.latency(threshold_ms=10000))
        pipeline.add_llm_grader(LLMAsJudge.correctness(threshold=JUDGE_THRESHOLD, model=DIRECT_JUDGE_MODEL))
        pipeline.add_llm_grader(LLMAsJudge.hallucination(threshold=JUDGE_THRESHOLD, model=DIRECT_JUDGE_MODEL))
        pipe_verdict = await pipeline.evaluate(
            session_summary=summary, trace_text=trace_text, final_response=answer, session_id=sid
        )
        golden_ref = golden_by_session.get(sid)
        trial_rep = await runner.run_trials(
            sid,
            golden_response=golden_ref,
            use_llm_judge=True,
            task_description="Answer the user's analytics question accurately.",
            thresholds=TRIAL_THRESHOLDS,
        )
        twin_eval_rows.append({
            "session_id": sid,
            "sql_golden_verdict": graded_verdict_by_session.get(sid, "UNGRADED"),
            "sdk_correctness": float(judge_summary_df.loc[sid, "correctness"]),
            "sdk_faithfulness": float(judge_summary_df.loc[sid, "faithfulness"]),
            "pipeline_score": round(pipe_verdict.final_score, 2),
            "pipeline_passed": bool(pipe_verdict.passed),
            "trial_pass@3": round(trial_rep.pass_at_k, 2),
            "trial_pass^3": round(trial_rep.pass_pow_k, 2),
        })
    except Exception as exc:
        print(f"Skipping direct Vertex AI judge evaluation for {sid}: {exc}")

twin_eval_df = pd.DataFrame(twin_eval_rows)
print("Side-by-side comparison of all quality evaluators on target sessions:")
display(twin_eval_df)

verdict_counts = graded_df.verdict.value_counts().to_dict()
twin_dict = twin_eval_df.set_index("session_id").to_dict(orient="index") if not twin_eval_df.empty else {}
RESULTS["s6"] = {
    "graded": len(graded_df),
    "verified_graded": len(verified_graded_df),
    "pass": int(verdict_counts.get("PASS", 0)),
    "fail": int(verdict_counts.get("FAIL", 0)),
    "ungraded": int(len(turns_df) - len(graded_df)),
    "failed_sessions": graded_df[graded_df.verdict == "FAIL"].session_id.tolist(),
    "corr_pass_rate": corr_pass_rate,
    "hall_pass_rate": hall_pass_rate,
    "twin_df": twin_dict,
}
if REFERENCE_SESSION in twin_dict and HALLUCINATED_SESSION in twin_dict:
    twin_summary_line = (
        f"- **Twin session separation (`{REFERENCE_SESSION}` vs. `{HALLUCINATED_SESSION}`)**: "
        f"Golden SQL grading (`{twin_dict[REFERENCE_SESSION]['sql_golden_verdict']}` vs. `{twin_dict[HALLUCINATED_SESSION]['sql_golden_verdict']}`), "
        f"`GraderPipeline` (**{twin_dict[REFERENCE_SESSION]['pipeline_score']:.2f}** vs. **{twin_dict[HALLUCINATED_SESSION]['pipeline_score']:.2f}**), "
        f"and `TrialRunner pass@3` judge consistency (**{twin_dict[REFERENCE_SESSION]['trial_pass@3']:.2f}** vs. **{twin_dict[HALLUCINATED_SESSION]['trial_pass@3']:.2f}**) "
        "all cleanly separate the accurate answer from the hallucinated one."
    )
else:
    twin_summary_line = f"- **Reference-free SDK judges**: `correctness` pass rate **{RESULTS['s6']['corr_pass_rate']:.0%}**, `hallucination` pass rate **{RESULTS['s6']['hall_pass_rate']:.0%}**."

show(f"""**Section 6 Takeaway (live)**

- **Golden Q&A SQL Grading (`AI.EMBED` + `AI.GENERATE`)**: Graded **{RESULTS['s6']['graded']}** in-domain sessions (**{RESULTS['s6']['pass']} PASS**, **{RESULTS['s6']['fail']} FAIL**; **{RESULTS['s6']['ungraded']}** out-of-domain/unkeyed sessions skipped). Failing session(s): {', '.join(f"`{s}`" for s in RESULTS['s6']['failed_sessions']) or 'none'}.
{twin_summary_line}""")

## 7. FinOps, latency SLOs, thinking token accounting & Verified Queries fast-path ROI

BQCA logs per-call `usage_metadata` (`prompt_token_count`, `candidates_token_count`, `thoughts_token_count`, `cached_content_token_count`, `total_token_count`) and latency (`total_ms`, `time_to_first_token_ms`). This section runs two FinOps workflows:

1. **Automated SLO Budget Checks (`SystemEvaluator`)**: Evaluates `latency`, `token_efficiency`, `cost_per_session`, `context_cache_hit_rate`, `turn_count`, and `ttft` via pure BigQuery SQL.
2. **Token & Thinking-Token Cost Estimation + Fast-Path ROI**: `SystemEvaluator.cost_per_session()` prices only `prompt_token_count` and `candidates_token_count`. Because Gemini reasoning models also generate `thoughts_token_count` (billed at the output token rate under token-based pricing), the query below computes a token-equivalent cost estimate at your configured `COST_RATES` including thinking tokens, and compares **Verified Queries Fast-Path (`is_fast_path = TRUE`)** against standard NL2SQL turns.

In [ ]:
#@title 7.1 Automated Session SLO Checks (SystemEvaluator)
BUDGETS = {
    "latency_ms": 3000,
    "total_tokens": 8000,
    "cost_usd": 0.02,
    "cache_hit_rate": 0.5,
    "turns": 3,
    "ttft_ms": 1000,
}
COST_RATES = {"input_per_1k": 0.00125, "output_per_1k": 0.01}  # Replace with your organization's Gemini rate card

SYSTEM_SUITE = [
    ("latency (avg event ms)", "latency", SystemEvaluator.latency(threshold_ms=BUDGETS["latency_ms"]), BUDGETS["latency_ms"]),
    ("token_efficiency (max tokens)", "token_efficiency", SystemEvaluator.token_efficiency(max_tokens=BUDGETS["total_tokens"]), BUDGETS["total_tokens"]),
    ("cost_per_session (max USD)", "cost", SystemEvaluator.cost_per_session(
        max_cost_usd=BUDGETS["cost_usd"], input_cost_per_1k=COST_RATES["input_per_1k"], output_cost_per_1k=COST_RATES["output_per_1k"]
    ), BUDGETS["cost_usd"]),
    ("context_cache_hit_rate (min)", "context_cache_hit_rate", SystemEvaluator.context_cache_hit_rate(min_hit_rate=BUDGETS["cache_hit_rate"]), BUDGETS["cache_hit_rate"]),
    ("turn_count (max turns)", "turn_count", SystemEvaluator.turn_count(max_turns=BUDGETS["turns"]), BUDGETS["turns"]),
    ("ttft (avg ms)", "ttft", SystemEvaluator.ttft(threshold_ms=BUDGETS["ttft_ms"]), BUDGETS["ttft_ms"]),
]

sys_rows = []
with timed("SystemEvaluator SLO suite via client.evaluate"):
    for label, metric, evaluator, budget in SYSTEM_SUITE:
        rep = client.evaluate(evaluator, filters=TraceFilter(limit=MAX_EVAL_SESSIONS))
        failing = sorted(s.session_id for s in rep.session_scores if not s.passed)
        sys_rows.append({
            "metric": metric,
            "evaluator": label,
            "budget": f"{budget:g}",
            "sessions": len(rep.session_scores),
            "passed": len(rep.session_scores) - len(failing),
            "failed": len(failing),
            "failing sessions": ", ".join(sid.replace("session_", "") for sid in failing) or "-",
        })
sys_df = pd.DataFrame(sys_rows)
display(sys_df.drop(columns="metric"))

In [ ]:
#@title 7.2 Thinking-Token Cost Accounting & Verified Queries Fast-Path ROI
finops_df = turns_df.copy()
rate_in, rate_out = COST_RATES["input_per_1k"], COST_RATES["output_per_1k"]
finops_df["cost_sdk_usd"] = finops_df.prompt_tokens / 1000.0 * rate_in + finops_df.candidate_tokens / 1000.0 * rate_out
finops_df["cost_incl_thinking_usd"] = finops_df.cost_sdk_usd + finops_df.thinking_tokens / 1000.0 * rate_out
finops_df["execution_path"] = finops_df.is_fast_path.astype(bool).map({
    True: "Fast Path (Verified Queries)",
    False: "Standard NL2SQL",
})

llm_turns_df = finops_df[finops_df.llm_calls > 0]
token_identity_ok = (
    llm_turns_df.prompt_tokens + llm_turns_df.candidate_tokens + llm_turns_df.thinking_tokens == llm_turns_df.total_tokens
)
print(f"Token accounting identity (prompt + candidates + thinking == total) holds on {int(token_identity_ok.sum())}/{len(llm_turns_df)} LLM sessions.")

finops_by_persona_df = (
    finops_df.groupby("persona")
    .agg(
        sessions=("session_id", "count"),
        llm_calls=("llm_calls", "sum"),
        prompt_tokens=("prompt_tokens", "sum"),
        candidate_tokens=("candidate_tokens", "sum"),
        thinking_tokens=("thinking_tokens", "sum"),
        cached_tokens=("cached_tokens", "sum"),
        total_tokens=("total_tokens", "sum"),
        cost_sdk_usd=("cost_sdk_usd", "sum"),
        cost_incl_thinking_usd=("cost_incl_thinking_usd", "sum"),
    )
    .round({"cost_sdk_usd": 4, "cost_incl_thinking_usd": 4})
    .reset_index()
)
denom_out = (finops_by_persona_df.candidate_tokens + finops_by_persona_df.thinking_tokens).replace(0, pd.NA)
finops_by_persona_df["thinking_share_of_output"] = (finops_by_persona_df.thinking_tokens / denom_out).fillna(0.0).round(2)
print("Token and estimated cost breakdown by user cohort (comparing SDK cost vs. cost including thinking tokens):")
display(finops_by_persona_df)

fast_path_cmp_df = (
    finops_df.groupby("execution_path")
    .agg(
        sessions=("session_id", "count"),
        llm_calls=("llm_calls", "sum"),
        e2e_min_ms=("e2e_latency_ms", "min"),
        e2e_p50_ms=("e2e_latency_ms", "median"),
        e2e_p90_ms=("e2e_latency_ms", lambda s: s.quantile(0.9)),
        median_total_tokens=("total_tokens", "median"),
        median_cost_incl_thinking_usd=("cost_incl_thinking_usd", "median"),
    )
    .round({"e2e_min_ms": 0, "e2e_p50_ms": 0, "e2e_p90_ms": 0, "median_total_tokens": 0, "median_cost_incl_thinking_usd": 4})
    .reset_index()
)
print("Verified Queries Fast-Path vs. Standard NL2SQL comparison:")
display(fast_path_cmp_df)

sdk_total_cost = float(finops_df.cost_sdk_usd.sum())
true_total_cost = float(finops_df.cost_incl_thinking_usd.sum())
thinking_add_pct = 100.0 * (true_total_cost - sdk_total_cost) / max(sdk_total_cost, 1e-9)
prompt_sum = float(llm_turns_df.prompt_tokens.sum())
weighted_cache_hit = float(llm_turns_df.cached_tokens.sum() / prompt_sum) if prompt_sum > 0 else 0.0
fc_idx = fast_path_cmp_df.set_index("execution_path")
fast_n = int(fc_idx.loc["Fast Path (Verified Queries)", "sessions"]) if "Fast Path (Verified Queries)" in fc_idx.index else 0
std_n = int(fc_idx.loc["Standard NL2SQL", "sessions"]) if "Standard NL2SQL" in fc_idx.index else 0
fast_p50_ms = float(fc_idx.loc["Fast Path (Verified Queries)", "e2e_p50_ms"]) if "Fast Path (Verified Queries)" in fc_idx.index else 0.0
std_p50_ms = float(fc_idx.loc["Standard NL2SQL", "e2e_p50_ms"]) if "Standard NL2SQL" in fc_idx.index else 0.0

sys_pass_map = {r.metric: (int(r.passed), int(r.sessions)) for r in sys_df.itertuples()}
failing_any_slo = sorted({s for val in sys_df["failing sessions"] if val != "-" for s in val.split(", ")})

RESULTS["s7"] = {
    "sys_pass": sys_pass_map,
    "failing_any": failing_any_slo,
    "identity_ok": int(token_identity_ok.sum()),
    "llm_sessions": len(llm_turns_df),
    "sdk_cost": sdk_total_cost,
    "true_cost": true_total_cost,
    "thinking_add_pct": thinking_add_pct,
    "weighted_cache_hit": weighted_cache_hit,
    "fast_n": fast_n,
    "std_n": std_n,
    "fast_p50_ms": fast_p50_ms,
    "std_p50_ms": std_p50_ms,
}
show(f"""**Section 7 Takeaway (live)**

- **SLO pass rates**: {'; '.join(f"`{k}` **{p}/{n}**" for k, (p, n) in RESULTS['s7']['sys_pass'].items())}. Sessions exceeding at least one budget: {', '.join(f"`{s}`" for s in RESULTS['s7']['failing_any']) or 'none'}.
- **Thinking-token cost accounting**: `prompt + candidates + thinking == total_token_count` holds on **{RESULTS['s7']['identity_ok']}/{RESULTS['s7']['llm_sessions']}** LLM sessions. Including `thoughts_token_count` raises estimated token-equivalent spend from **${RESULTS['s7']['sdk_cost']:.3f}** to **${RESULTS['s7']['true_cost']:.3f}** (**+{RESULTS['s7']['thinking_add_pct']:.0f}%** over the `cost_per_session` estimate); token-weighted context cache hit rate is **{RESULTS['s7']['weighted_cache_hit']:.2f}**.
- **Verified Queries Fast-Path ROI**: Fast-path turns (`n={RESULTS['s7']['fast_n']}`) achieved a median end-to-end latency of **{RESULTS['s7']['fast_p50_ms']:,.0f} ms** with **0** LLM tokens, compared with **{RESULTS['s7']['std_p50_ms']:,.0f} ms** across standard NL2SQL turns (`n={RESULTS['s7']['std_n']}`).""")

## 8. Operational health monitor (`finish_reason`, sanitised `*_ERROR` rows & polite refusals) + Executive Summary

In BQCA production logs, operational issues appear across three distinct signals:

1. **Model generation cut-offs (`LLM_RESPONSE.attributes.finish_reason`)**: Detects responses truncated by `MAX_TOKENS` or blocked by `SAFETY` even when no `*_ERROR` event was emitted.
2. **Sanitised error events (`INVOCATION_ERROR`, `AGENT_ERROR`, `LLM_ERROR`)**: Query these directly with `bqaa.ERROR_SQL_PREDICATE` (`status = 'ERROR' OR error_message IS NOT NULL OR ENDS_WITH(event_type, '_ERROR')`). Do **not** rely on `SystemEvaluator.error_rate()` on BQCA logs, because `SystemEvaluator.error_rate()` only counts `TOOL_ERROR` events (which BQCA omits by design).
3. **Polite out-of-scope refusals (`AGENT_RESPONSE`)**: When a user asks an out-of-domain or policy-restricted question, the agent answers politely with `status = 'OK'` and `finish_reason = 'STOP'`. Use `evaluate_categorical()` to separate substantive answers from polite refusals, clarifying questions, and unanswered error turns.

In [ ]:
#@title 8.1 Model Finish Reasons & Sanitised Error Events (bqaa.ERROR_SQL_PREDICATE)
finish_df = run_sql(f"""
SELECT
  COALESCE(NULLIF(SPLIT(user_id, '@')[SAFE_OFFSET(0)], ''), 'unattributed') AS user_cohort,
  session_id,
  COALESCE(JSON_VALUE(attributes, '$.finish_reason'), 'STOP') AS finish_reason,
  SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.candidates_token_count') AS INT64) AS candidate_tokens
FROM {T(COMPAT_VIEW_ID)}
WHERE event_type = 'LLM_RESPONSE'
""")
finish_df["persona"] = finish_df["session_id"].map(PERSONA_BY_SESSION).fillna(finish_df["user_cohort"])
finish_df = finish_df[["persona", "session_id", "finish_reason", "candidate_tokens"]]

print("1. Model finish_reason distribution across all LLM_RESPONSE events:")
display(
    finish_df.groupby("finish_reason")
    .agg(llm_responses=("session_id", "count"), sessions=("session_id", "nunique"))
    .reset_index()
    .sort_values("llm_responses", ascending=False)
    .reset_index(drop=True)
)
non_stop_df = finish_df[finish_df.finish_reason != "STOP"].sort_values(["finish_reason", "session_id"]).reset_index(drop=True)
print("Sessions with a non-STOP model finish_reason:")
display(non_stop_df)

errors_df = run_sql(f"""
SELECT
  COALESCE(NULLIF(SPLIT(user_id, '@')[SAFE_OFFSET(0)], ''), 'unattributed') AS user_cohort,
  session_id,
  event_type,
  status,
  content IS NULL AS content_is_null,
  error_message
FROM {T(COMPAT_VIEW_ID)}
WHERE {bqaa.ERROR_SQL_PREDICATE}
ORDER BY session_id, timestamp
""")
errors_df["persona"] = errors_df["session_id"].map(PERSONA_BY_SESSION).fillna(errors_df["user_cohort"])
errors_df = errors_df[["persona", "session_id", "event_type", "status", "content_is_null", "error_message"]]


def classify_error_message(msg):
    m = (msg or "").lower()
    if "resource_exhausted" in m or "429" in m:
        return "Quota / Rate Limit (429)"
    if "query failed" in m:
        return "SQL Query Execution Error"
    return "Other Runtime Error"


errors_df["error_class"] = errors_df.error_message.map(classify_error_message)
print(f"\n2. Sanitised error rows matched by bqaa.ERROR_SQL_PREDICATE ({len(errors_df)} rows across {errors_df.session_id.nunique()} sessions):")
display(errors_df)

In [ ]:
#@title 8.2 Delivery Outcome Classification & Operational Attention Queue
outcome_metric = CategoricalMetricDefinition(
    name="delivery_outcome",
    definition="What the agent delivered to the user in this session.",
    categories=[
        CategoricalMetricCategory(
            name="Answered",
            definition="The agent delivered a substantive data answer to the user's question.",
        ),
        CategoricalMetricCategory(
            name="Refused_Out_of_Scope",
            definition="The agent politely declined because the request is outside the available dataset or data-governance policy.",
        ),
        CategoricalMetricCategory(
            name="Clarification_Requested",
            definition="The agent asked the user a clarifying question to disambiguate the request.",
        ),
        CategoricalMetricCategory(
            name="No_Answer_Delivered",
            definition="The session terminated with an error before any answer was delivered.",
        ),
    ],
)
outcome_cfg = CategoricalEvaluationConfig(
    metrics=[outcome_metric],
    endpoint=CATEGORICAL_MODEL,
    connection_id=VERTEX_CONNECTION_ID,
    include_justification=True,
)
with timed(f"evaluate_categorical: delivery_outcome ({CATEGORICAL_MODEL})"):
    outcome_report = client.evaluate_categorical(outcome_cfg, filters=TraceFilter(limit=MAX_EVAL_SESSIONS))
print("Execution mode:", outcome_report.details.get("execution_mode"))
outcome_df = cat_frame(outcome_report)

# Combine structured failure signals (has_error, non-STOP finish_reason) with text classification
non_stop_by_session = non_stop_df.groupby("session_id").finish_reason.agg(lambda s: ", ".join(sorted(set(s))))
ops_df = (
    turns_df[["session_id", "persona", "has_error", "error_messages"]]
    .merge(outcome_df[["session_id", "category", "justification"]], on="session_id", how="left")
)
ops_df["category"] = ops_df["category"].fillna("Not_Evaluated")
ops_df["justification"] = ops_df["justification"].fillna("-")
ops_df["non_stop_finish"] = ops_df["session_id"].map(non_stop_by_session).fillna("-")
ops_df["has_structured_alert"] = ops_df["has_error"].astype(bool) | (ops_df["non_stop_finish"] != "-")

print("Delivery outcome cross-tabulated by user cohort:")
display(pd.crosstab(ops_df["persona"], ops_df["category"]))

attention_mask = (
    ops_df["category"].isin(["Refused_Out_of_Scope", "Clarification_Requested", "No_Answer_Delivered"])
    | ops_df["has_structured_alert"]
)
print("Operational attention queue (sessions with an error row, non-STOP finish_reason, refusal, or clarification):")
display(
    ops_df[attention_mask][
        ["persona", "session_id", "category", "has_error", "non_stop_finish", "justification"]
    ].sort_values(["category", "persona", "session_id"]).reset_index(drop=True)
)

RESULTS["s8"] = {
    "finish_counts": finish_df.finish_reason.value_counts().to_dict(),
    "non_stop_sessions": dict(zip(non_stop_df.session_id, non_stop_df.finish_reason)),
    "error_rows": len(errors_df),
    "error_sessions": int(errors_df.session_id.nunique()),
    "all_error_content_null": bool(errors_df.content_is_null.all()) if not errors_df.empty else True,
    "outcomes": outcome_df.category.value_counts().to_dict(),
    "refusals": int((outcome_df.category == "Refused_Out_of_Scope").sum()),
}

### 8.3 Customer Production Health & Analytics Summary Table (Live)

The summary table below is computed dynamically from the live BigQuery queries and SDK evaluations executed in Sections 1–8.

In [ ]:
#@title 8.3 Customer Production Health & Analytics Summary Table
R = RESULTS
twin_summary_cell = (
    f"; `GraderPipeline` (**{R['s6']['twin_df'][REFERENCE_SESSION]['pipeline_score']:.2f}** vs. **{R['s6']['twin_df'][HALLUCINATED_SESSION]['pipeline_score']:.2f}**) "
    f"and `TrialRunner pass@3` (**{R['s6']['twin_df'][REFERENCE_SESSION]['trial_pass@3']:.2f}** vs. **{R['s6']['twin_df'][HALLUCINATED_SESSION]['trial_pass@3']:.2f}**) separated the twin sessions."
    if REFERENCE_SESSION in R["s6"]["twin_df"] and HALLUCINATED_SESSION in R["s6"]["twin_df"]
    else f"; SDK `correctness` pass rate **{R['s6']['corr_pass_rate']:.0%}**, `hallucination` pass rate **{R['s6']['hall_pass_rate']:.0%}**."
)

summary_rows = [
    (
        "1. Views & Schema Health",
        f"`{COMPAT_VIEW_ID}`, `{TURNS_VIEW_ID}`, `client.doctor()`",
        f"Schema `{R['s1']['schema_status']}` ({R['s1']['columns']} cols); **{R['s1']['total_events']}** events across **{R['s2']['sessions']}** sessions and **{R['s1']['active_event_types']}** BQCA event types; **{R['s1']['scalar_responses']}/{R['s1']['agent_response_rows']}** `AGENT_RESPONSE` rows normalized.",
    ),
    (
        "2. Conversation Explorer",
        f"`{TURNS_VIEW_ID}` SQL",
        f"**{R['s2']['answered']}/{R['s2']['sessions']}** sessions delivered an answer (including **{R['s2']['clarifying']}** clarifying question); **{R['s2']['fast_path']}** hit Verified Queries fast path; **{R['s2']['errors']}** ended in an error.",
    ),
    (
        "3. Sentiment & UX Triage",
        f"`LLMAsJudge.sentiment()`, `evaluate_categorical()` (`{JUDGE_MODEL}`)",
        f"**{R['s3']['passed']}/{R['s3']['sessions']}** sessions passed the `{JUDGE_THRESHOLD}` sentiment bar; UX rubric flagged **{len(R['s3']['frustrated_sessions'])}** `Frustrated_Impatient` (`{', '.join(R['s3']['frustrated_sessions']) or 'none'}`) and **{len(R['s3']['clarification_sessions'])}** `Clarification_Needed`.",
    ),
    (
        "4. Question Topic Taxonomy",
        f"`evaluate_categorical()` (`AI.CLASSIFY` & `AI.GENERATE`, `{CATEGORICAL_MODEL}`)",
        f"`AI.CLASSIFY` and `AI.GENERATE` agreed on **{R['s4']['agree']}/{R['s4']['sessions']}** sessions ({', '.join(f'{k}: {v}' for k, v in R['s4']['distribution'].items())}).",
    ),
    (
        "5. Semantic Drift & Coverage",
        "`client.drift_detection()`, `AI.EMBED` + `ML.DISTANCE`",
        f"**{R['s5']['coverage_pct']:.1f}%** Golden Question Bank coverage (**{R['s5']['covered_golden']}/{R['s5']['total_golden']}** covered); **{R['s5']['drifted_over_015']}** production sessions drifted beyond cosine distance `{DRIFT_THRESHOLD}` (top drifted cohort: `{R['s5']['top_drift_persona']}`, median `{R['s5']['top_drift_median']:.3f}`).",
    ),
    (
        "6. Response Quality & Hallucination Catch",
        f"Golden Q&A `AI.GENERATE` grader (`{JUDGE_MODEL}`), `LLMAsJudge`, `GraderPipeline`, `TrialRunner`",
        f"Golden Q&A grading: **{R['s6']['pass']} PASS** and **{R['s6']['fail']} FAIL** across **{R['s6']['graded']}** in-domain sessions (caught `{', '.join(R['s6']['failed_sessions']) or 'none'}`){twin_summary_cell}",
    ),
    (
        "7. FinOps, Thinking Tokens & Fast Path",
        f"`SystemEvaluator.*`, `{TURNS_VIEW_ID}` token accounting",
        f"Latency SLO **{R['s7']['sys_pass']['latency'][0]}/{R['s7']['sys_pass']['latency'][1]}**, token SLO **{R['s7']['sys_pass']['token_efficiency'][0]}/{R['s7']['sys_pass']['token_efficiency'][1]}**, cache hit rate **{R['s7']['weighted_cache_hit']:.2f}**; including `thoughts_token_count` adds **+{R['s7']['thinking_add_pct']:.0f}%** to SDK cost (**${R['s7']['sdk_cost']:.3f}** -> **${R['s7']['true_cost']:.3f}**); Fast-Path median latency **{R['s7']['fast_p50_ms']:,.0f} ms** vs. **{R['s7']['std_p50_ms']:,.0f} ms**.",
    ),
    (
        "8. Operational Health Monitor",
        "`finish_reason`, `bqaa.ERROR_SQL_PREDICATE`, `evaluate_categorical()`",
        f"**{R['s8']['error_rows']}** sanitised error rows in **{R['s8']['error_sessions']}** sessions (`content IS NULL`: `{R['s8']['all_error_content_null']}`); **{len(R['s8']['non_stop_sessions'])}** non-STOP `finish_reason` sessions (`{R['s8']['non_stop_sessions']}`); **{R['s8']['refusals']}** polite out-of-scope refusals.",
    ),
]

show(
    "| Section & Workflow | SDK / BigQuery Surface | Live Findings on This Dataset |\n|---|---|---|\n"
    + "\n".join(f"| **{a}** | {b} | {c} |" for a, b, c in summary_rows)
)